# Pelajaran 08 - Pola Desain Multi-Agen


## Pengaturan


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mengapa Sistem Multi-Agen?

Tugas dunia nyata seperti perencanaan perjalanan melibatkan berbagai jenis keahlian — logistik, pengetahuan lokal, penganggaran, dan lainnya. Satu agen yang mencoba menangani semuanya dengan cepat menjadi sulit diatur.

Sistem multi-agen menyelesaikan ini melalui **spesialisasi**: setiap agen fokus pada satu bidang keahlian, menghasilkan hasil berkualitas lebih tinggi daripada seorang generalis. Mereka juga meningkatkan **skalabilitas** — Anda dapat menambahkan agen baru (misalnya, spesialis penerbangan, kritikus restoran) tanpa menulis ulang alur kerja yang ada. Agen-agen tersebut bekerja bersama melalui sebuah alur kerja terstruktur, meneruskan konteks dari satu agen ke agen berikutnya.


## Membuat Agen Khusus


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Membangun Alur Kerja Berurutan

`WorkflowBuilder` memungkinkan Anda menghubungkan agen ke dalam grafik berarah. Di sini kita membuat pipeline dua langkah sederhana: **TravelPlanner** menyusun rencana perjalanan, kemudian **TravelConcierge** meninjau dan menyempurnakannya.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Menambahkan Lebih Banyak Agen ke Alur Kerja

Salah satu keuntungan terbesar dari pola multi-agen adalah betapa mudahnya untuk diperluas. Di bawah ini kami menambahkan agen **BudgetReviewer** yang memeriksa rencana terhadap anggaran pelancong, menandai item yang mungkin mendorong biaya melewati batas, dan menyarankan alternatif penghematan uang. Alur kerja sekarang menjalankan tiga agen secara berurutan:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Ringkasan

Dalam pelajaran ini Anda belajar bagaimana:

1. **Membuat agen khusus** — masing-masing dengan peran yang fokus (perencanaan, resepsionis, tinjauan anggaran).
2. **Menghubungkan agen ke dalam alur kerja berurutan** menggunakan `WorkflowBuilder` dan `add_edge`.
3. **Mengalirkan output** dari jalur multi-agen, melacak agen mana yang sedang berbicara.
4. **Memperluas alur kerja** dengan menambahkan agen baru ke rantai tanpa mengubah agen yang sudah ada.

Pola desain multi-agen menjaga setiap agen tetap sederhana sekaligus menghasilkan hasil yang lebih kaya dan lebih teliti ditinjau dibandingkan apa yang dapat dicapai oleh satu agen saja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sahih. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau salah tafsir yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
